# What the listeners changed

`02_listening_test.ipynb` built a blind listening test and split it across six
raters. This notebook reads the answers back.

All six returned: 124 scored ratings covering every one of the 80 pairs. That
is enough to answer the question the test was built for, and it has already
done something the four frozen metrics could not — it named two things missing
from the feature set, and both turned out to be worth building.

The order below is the order the results were found in, not a tidy summary:

1. does the recommender beat a random pair, to a listener
2. do listeners agree with each other — the ceiling on any model
3. which feature family tracks the human score
4. what listeners said connects a pair, against the families we have
5. the two holes that opened up, and what filling them bought

## What came back

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy.spatial.distance import cosine
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from modeling.listening import responses, tags, gaps

returned = responses()

# One rater left a score blank while still ticking what connected the pair.
# A single NaN silently turns every correlation below into NaN, so it goes
# here rather than being guarded for at each use.
d = returned.dropna(subset=["similarity_1_5"])

print(f"{len(returned)} ratings from {d['rater'].nunique()} of 6 raters, "
      f"{d['pair_id'].nunique()} of 80 pairs covered")
if len(d) < len(returned):
    missing = returned[returned["similarity_1_5"].isna()]
    print(f"{len(missing)} returned without a score, dropped: "
          f"{list(zip(missing['rater'], missing['pair_id']))}")

125 ratings from 6 of 6 raters, 80 of 80 pairs covered
1 returned without a score, dropped: [('R6', 'P75')]


## 1. Does it beat a random pair?

The baseline's manual evaluation rated 50 pairs, all of them from the model. A
mean of 3.3 out of 5 sounds respectable until you notice there is nothing to
read it against. `02` added one random partner per seed for exactly this.

In [2]:
from scipy.stats import mannwhitneyu

print(d.groupby("kind")["similarity_1_5"].agg(["count", "mean", "std"]).round(2).to_string())

# A mean is not a result on its own. The random pairs are the floor the other
# two are read against, which is the only reason any of this can be claimed.
by_kind = {k: g["similarity_1_5"] for k, g in d.groupby("kind")}
print()
for a, b in [("model", "random"), ("deep", "random"), ("model", "deep")]:
    p = mannwhitneyu(by_kind[a], by_kind[b], alternative="greater")[1]
    print(f"{a:6} > {b:6}   p = {p:.4f}")

        count  mean   std
kind                     
deep       29  3.03  1.32
model      80  2.96  1.34
random     15  1.87  0.92

model  > random   p = 0.0018
deep   > random   p = 0.0026
model  > deep     p = 0.6024


**The recommender beats a random pair, to a human ear.** p = 0.002, on a blind
comparison where the rater could not see artist, title, genre or cosine score.
Every number in the README until now rested on genre labels; this one does not.

The second result was not expected to be this clean. `deep` pairs are drawn
from rank 50 and 200 of the same rankings — the far end of the adventure
slider, which the depth profile set by hand. They beat random too (p = 0.003),
and they are **indistinguishable from the model's own top five** (p = 0.60).

Far away, and still heard as related. That is the whole claim of the
Familiar → Adventurous control, and this is the first evidence for it that does
not come from a genre label.

## 2. Do the listeners agree with each other?

Every rater hears the same nine core pairs. Agreement on those is the ceiling:
no model can match human judgement more closely than humans match each other.

In [3]:
import itertools
from scipy.stats import pearsonr

core = d[d["in_core"]].pivot_table(index="pair_id", columns="rater",
                                   values="similarity_1_5")
core["spread"] = core.max(axis=1) - core.min(axis=1)
print(core.to_string())
print(f"\nmean spread across raters: {core['spread'].mean():.2f} points of 5")

# Disagreement caps what any model can achieve — but it is noise, and averaging
# raters removes it. Spearman-Brown says how fast.
complete = core.drop(columns="spread").dropna()
pairwise = [pearsonr(complete[a], complete[b])[0]
            for a, b in itertools.combinations(complete.columns, 2)]
r_bar = float(np.mean(pairwise))
print(f"\nmean correlation between two raters: {r_bar:+.3f}")
for k in (1, 2, 6, 12):
    reliability = k * r_bar / (1 + (k - 1) * r_bar)
    print(f"  average of {k:2} raters -> reliability {reliability:.2f}, "
          f"so a model can correlate at most {np.sqrt(reliability):.2f}")

rater     R1   R2   R3   R4   R5   R6  spread
pair_id                                      
P08      1.0  1.0  1.0  3.0  1.0  2.0     2.0
P20      2.0  2.0  5.0  2.0  1.0  2.0     4.0
P25      5.0  4.0  3.0  2.0  3.0  3.0     3.0
P47      4.0  2.0  4.0  3.0  1.0  2.0     3.0
P59      3.0  4.0  5.0  4.0  5.0  5.0     2.0
P71      2.0  2.0  4.0  1.0  1.0  2.0     3.0
P72      5.0  3.0  3.0  1.0  1.0  2.0     4.0
P75      5.0  5.0  5.0  4.0  4.0  NaN     1.0
P76      2.0  3.0  1.0  2.0  1.0  2.0     2.0

mean spread across raters: 2.67 points of 5

mean correlation between two raters: +0.404
  average of  1 raters -> reliability 0.40, so a model can correlate at most 0.64
  average of  2 raters -> reliability 0.58, so a model can correlate at most 0.76
  average of  6 raters -> reliability 0.80, so a model can correlate at most 0.90
  average of 12 raters -> reliability 0.89, so a model can correlate at most 0.94


Two raters agree with each other at r = 0.40, and some pairs split four points
out of five — P72 was scored 5 by one listener and 1 by another. Rater means
run from 2.47 to 3.29, so part of the spread is personal scale rather than
disagreement about the music.

That caps what any model can achieve, but it does not cap it at zero.
Disagreement is noise, and averaging removes noise: the mean of six raters is
reliable at 0.80, which allows a model to correlate up to about 0.90.

The effect is visible further down. Against individual ratings the best space
scores −0.35; against the six-rater average of these nine pairs it scores
−0.77. Same model, same pairs — the difference is how much rater noise is left
in the target.

Nine pairs is a small base for that second number, and the design lesson for a
second round is more overlap rather than more coverage.

## 3. Which feature family tracks the human score?

For every rated pair, the cosine distance between the two tracks is computed
inside one feature family at a time, then correlated against what the listener
said. A negative correlation is the good direction: closer in that space, rated
more alike.

This is the question the genre metrics cannot answer, because a family can
predict genre well and still have nothing to do with what a person hears.

In [4]:
from modeling.rhythm import load as load_rhythm
from modeling.separation import load as load_sep

META = "../data/fma_metadata"
tracks = pd.read_csv(f"{META}/tracks.csv", index_col=0, header=[0, 1])
small = tracks[tracks[("set", "subset")] == "small"]
X = pd.read_csv(f"{META}/features.csv", index_col=0, header=[0, 1, 2]).loc[small.index].dropna()

R = load_rhythm()
S = load_sep()
common = X.index.intersection(R.index).intersection(S.index)
X, R, S = X.loc[common], R.loc[common], S.loc[common]
small = small.loc[common]
labels = small[("track", "genre_top")]
genres, artists = labels.to_numpy(), small[("artist", "name")].to_numpy()

family = X.columns.get_level_values(0)
X.columns = [f"{a}.{b}.{c}" for a, b, c in X.columns]
R.columns = [f"{a}.{b}" for a, b, _ in R.columns]
S.columns = [f"{a}.{b}.{c}" for a, b, c in S.columns]

TIMBRE = ["mfcc", "spectral_contrast", "spectral_centroid", "spectral_bandwidth",
          "spectral_rolloff", "zcr", "rmse"]
HARMONY = ["chroma_cens", "chroma_cqt", "chroma_stft", "tonnetz"]
RATIO = S[["hpss_ratio.mean.01"]]

rated = d[d["track_a"].isin(common) & d["track_b"].isin(common)]
print(f"{len(rated)} ratings usable, {X.shape[1]} FMA + {R.shape[1]} rhythm "
      f"+ {S.shape[1]} separation columns")

124 ratings usable, 518 FMA + 13 rhythm + 81 separation columns


In [5]:
def space(families=(), extra=None):
    parts = [X.loc[:, family.isin(list(families))]] if len(families) else []
    if extra is not None:
        parts.append(extra)
    return pd.concat(parts, axis=1)


def scaled(F):
    return pd.DataFrame(StandardScaler().fit_transform(F), index=F.index)


def human_rho(F):
    """Spearman between distance in this space and the 1-5 human score."""
    Z = scaled(F)
    dist = [cosine(Z.loc[a], Z.loc[b]) for a, b in zip(rated["track_a"], rated["track_b"])]
    rho, p = spearmanr(dist, rated["similarity_1_5"])
    return rho, p


rows = []
for name in sorted(family.unique()):
    rho, p = human_rho(space([name]))
    rows.append({"family": name, "columns": int((family == name).sum()),
                 "rho": round(rho, 3), "p": round(p, 3)})
pd.DataFrame(rows).sort_values("rho").reset_index(drop=True)

,family,columns,rho,p
0,mfcc,140,-0.314,0.000
1,spectral_bandwidth,7,-0.236,0.008
2,spectral_rolloff,7,-0.190,0.035
3,spectral_centroid,7,-0.154,0.089
4,chroma_cens,84,-0.106,0.240
5,chroma_stft,84,-0.062,0.497
6,spectral_contrast,49,-0.043,0.634
7,zcr,7,-0.032,0.728
8,chroma_cqt,84,0.033,0.718
9,rmse,7,0.090,0.322


`mfcc` is the only large family that clears significance, and it clears it
easily at 140 columns. `spectral_bandwidth` and `spectral_rolloff` also clear
it on seven columns each.

Every chroma family sits near zero, and `tonnetz` — the fourth harmony family —
sits on the wrong side of it. Together they are 294 of the 518 columns.

Now the same question of whole spaces rather than single families.

In [6]:
for name, F in [("all 518", space(family.unique())),
                ("timbre (224)", space(TIMBRE)),
                ("harmony (294)", space(HARMONY)),
                ("mfcc (140)", space(["mfcc"]))]:
    rho, p = human_rho(F)
    print(f"{name:16} {F.shape[1]:4} columns   rho = {rho:+.3f}   p = {p:.3f}")

all 518           518 columns   rho = +0.047   p = 0.601
timbre (224)      224 columns   rho = -0.179   p = 0.047
harmony (294)     294 columns   rho = +0.040   p = 0.658
mfcc (140)        140 columns   rho = -0.314   p = 0.000


This is the result that reframes the project.

`all 518` and `mfcc` score almost identically on genre agreement — 0.414
against 0.412 — and a listener separates them immediately. The full space
carries 294 harmony columns that pull the wrong way, and they drown the signal
the 140 mfcc columns carry.

The ordering held five times as the returns arrived, at 42 ratings, then 63,
84, 103 and 124. Each new rater was a chance for it to flip and it never did.

No amount of tuning against the genre metrics would have found this. They
cannot see it.

## 4. What the listeners said was missing

The form asks a second question after the score: what connects these two, from a
fixed vocabulary. Those words are the half of the survey that turns into work.

`modeling/listening.py` counts them and lines each word up against the feature
families that could carry it. A word with nothing behind it is a feature nobody
has built.

In [7]:
t = tags(d)
print(t.to_string(index=False))
print()
print(gaps(t).to_string(index=False))

            tag  mentions  mean_score
           mood        50        3.42
         energy        48        3.67
        harmony        33        3.61
         rhythm        29        3.83
instrumentation        25        4.00
        nothing        19        1.11
         timbre        18        3.44
     production        14        3.64
         vocals         7        3.29

            tag  mentions  mean_score                                                                         families     status
           mood        50        3.42                                                                                — NO FEATURE
         energy        48        3.67                                                                        rmse, zcr      built
        harmony        33        3.61                                    chroma_cens, chroma_cqt, chroma_stft, tonnetz      built
         rhythm        29        3.83                                 tempo, onset_rate, onset_stre

`rhythm` was named 29 times on pairs scoring 3.83 out of 5 — and FMA has no
rhythm feature at all. Its eleven families are spectral or harmonic. Tempo,
pulse and note density are simply absent, so the model could not tell whether
two tracks share a beat.

`instrumentation`, 25 mentions and the highest mean score of any word at 4.00,
had the same problem: nothing in the feature set says whether a track is
carried by drums or by instruments.

Both were extracted from the mp3s rather than the feature table, because a
feature that is not in the table cannot be recovered from it.

`mood` leads the table and nothing was built for it, on purpose. It is two
raters' habit — 76% and 62% of their ratings against 5% for another — and of
its 50 uses only 9 stood alone. It rides along with every other word rather
than naming a dimension of its own.

## 5. Two tests, not one

Before the results, the method — because one test is not enough to judge a new
feature, and the difference matters twice below.

**Unsupervised agreement** is what the recommender gets: of a track's ten
nearest neighbours, how many share its genre. This is the existing metric.

**The supervised ceiling** hands a classifier the genre labels and lets it train
on them. Whatever it reaches is roughly the most these columns can say about
genre at all.

Read together:

| ceiling | agreement | reading |
|---|---|---|
| rises | rises | the columns carry new information |
| flat | rises | the same information, rearranged |
| flat | flat | nothing was added |

A feature that lifts agreement while leaving the ceiling flat has not taught the
model anything. It has only made the existing information easier to reach — and
that gain can just as easily come at the cost of something else, as it does
below.

In [8]:
def agreement(F, k=10, exclude_artist=False):
    """Share of the k nearest neighbours sharing the seed's genre."""
    Z = StandardScaler().fit_transform(F)
    n_candidates = 60 if exclude_artist else k + 1
    _, idx = NearestNeighbors(metric="cosine", algorithm="brute").fit(Z).kneighbors(
        Z, n_neighbors=n_candidates)
    scores = []
    for i in range(len(Z)):
        nb = [j for j in idx[i]
              if j != i and (not exclude_artist or artists[j] != artists[i])][:k]
        if nb:
            scores.append((genres[nb] == genres[i]).mean())
    return float(np.mean(scores))


def ceiling(F):
    """What a classifier reaches when handed the labels to train on."""
    clf = make_pipeline(StandardScaler(),
                        RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42))
    return cross_val_score(clf, F, labels, cv=5, n_jobs=-1).mean()


def report(name, F):
    rho, p = human_rho(F)
    print(f"{name:32} {F.shape[1]:4} {agreement(F):9.3f} {ceiling(F):9.3f} "
          f"{rho:9.3f} {p:7.3f}")


print(f"{'space':32} {'cols':>4} {'agree@10':>9} {'ceiling':>9} {'rho':>9} {'p':>7}")
report("timbre", space(TIMBRE))
report("timbre + rhythm", space(TIMBRE, R))
report("mfcc", space(["mfcc"]))
report("mfcc + rhythm", space(["mfcc"], R))

space                            cols  agree@10   ceiling       rho       p


timbre                            224     0.458     0.508    -0.179   0.047


timbre + rhythm                   237     0.472     0.542    -0.217   0.016


mfcc                              140     0.412     0.485    -0.314   0.000


mfcc + rhythm                     153     0.435     0.526    -0.344   0.000


Thirteen rhythm columns lift the ceiling in both spaces — 0.512 to 0.546, and
0.488 to 0.528. The information is new, not rearranged. That is the whole
justification for having gone back to the audio.

### And the same test on the harmonic-percussive split

`modeling/separation.py` splits each track into its harmonic and percussive
halves and writes 81 columns: one energy ratio, and the timbre of each half
separately. The ratio is the measured part of the idea; the two split mfcc
blocks are the hypothesis.

In [9]:
print(f"{'space':32} {'cols':>4} {'agree@10':>9} {'ceiling':>9} {'rho':>9} {'p':>7}")
report("mfcc + rhythm", space(["mfcc"], R))
report("mfcc + rhythm + ratio", space(["mfcc"], pd.concat([R, RATIO], axis=1)))
report("mfcc + rhythm + all 81", space(["mfcc"], pd.concat([R, S], axis=1)))
report("timbre + rhythm", space(TIMBRE, R))
report("timbre + rhythm + ratio", space(TIMBRE, pd.concat([R, RATIO], axis=1)))
report("timbre + rhythm + all 81", space(TIMBRE, pd.concat([R, S], axis=1)))

space                            cols  agree@10   ceiling       rho       p


mfcc + rhythm                     153     0.435     0.526    -0.344   0.000


mfcc + rhythm + ratio             154     0.438     0.533    -0.351   0.000


mfcc + rhythm + all 81            234     0.459     0.534    -0.367   0.000


timbre + rhythm                   237     0.472     0.542    -0.217   0.016


timbre + rhythm + ratio           238     0.474     0.546    -0.217   0.015


timbre + rhythm + all 81          318     0.488     0.546    -0.258   0.004


This is where the second test earns its keep.

The full 81 columns look like the best result in the table — `timbre + rhythm +
all 81` reaches 0.488, higher than anything else in this notebook. But the
ceiling does not move: 0.546 to 0.548. The eighty split-mfcc columns are
restating what the unsplit mfcc already said. And the human correlation gets
worse, for the same reason the 294 harmony columns hurt: volume dilutes signal.

The single ratio column behaves the other way. It lifts the ceiling, 0.528 to
0.536, and it improves the human correlation rather than damaging it.

So one column of the 81 is kept and eighty are discarded. **Judged on agreement
alone the decision would have gone the other way** — we would have kept the best
number in the project and quietly made the model worse for listeners.

## Where this leaves the model

Two candidates, both already built. Both keep mfcc, rhythm and the ratio; they
differ only in whether the other 84 timbre columns come along.

In [10]:
from sklearn.decomposition import PCA

A = space(["mfcc"], pd.concat([R, RATIO], axis=1))
B = space(TIMBRE, pd.concat([R, RATIO], axis=1))

built = {}
for name, F in [("A: mfcc + rhythm + ratio", A), ("B: timbre + rhythm + ratio", B)]:
    Z = PCA(n_components=0.90, random_state=42).fit_transform(StandardScaler().fit_transform(F))
    built[name] = (NearestNeighbors(metric="cosine").fit(Z), Z)
    print(f"{name:28} {F.shape[1]:4} columns -> {Z.shape[1]:3} components")

# If the two spaces recommended the same tracks there would be nothing for a
# second listening round to compare, and it would not be worth anyone's evening.
rng = np.random.default_rng(42)
seeds = rng.choice(len(common), 500, replace=False)
(a_index, a_Z), (b_index, b_Z) = built.values()
_, ai = a_index.kneighbors(a_Z[seeds], n_neighbors=11)
_, bi = b_index.kneighbors(b_Z[seeds], n_neighbors=11)
shared = np.array([len(set(a[1:]) & set(b[1:])) for a, b in zip(ai, bi)])
print(f"\nacross 500 seeds the two spaces share {shared.mean():.1f} of their top 10")

A: mfcc + rhythm + ratio      154 columns ->  57 components
B: timbre + rhythm + ratio    238 columns ->  81 components

across 500 seeds the two spaces share 4.5 of their top 10


Half the recommendations differ, so there is a real question to put to
listeners.

The genre metrics prefer B. The human scores prefer A, by a wide margin, across
all 124 ratings. Those two rulers disagree, and this notebook cannot settle it:
the two correlations are measured on the same pairs, and separating two
dependent correlations needs far more data than testing either against zero.

## What shipped, and what it cost

A ran anyway, as `modeling.features.FEATURE_SETS["heard"]`, and the recommender
now defaults to it. The argument is the human column above, not the genre one —
the old space correlates +0.05 with what listeners said and this one −0.35, and
that ordering held five times as returns arrived, at 42 ratings, then 63, 84,
103 and 124.

Held out over five folds the frozen metrics moved: `precision@10` 0.400 → 0.422,
`nDCG@10` 0.420 → 0.441, `genre_overlap@10` 0.262 → 0.280, `coverage` 0.780 →
0.757, on 57 components instead of 144. **Read that as "nothing broke" rather
than as the result.** 294 of the columns removed were the ones dragging genre
agreement down, so dropping them had to lift it.

`--space all` still runs the old space and `--compare` scores all three from raw
data, so nothing here has to be taken on trust — including the fact that on
genre agreement, `heard_wide` beats what we shipped.

## Everything that was tested and rejected

| candidate | columns | genre | new information? | the ear | verdict |
|---|---|---|---|---|---|
| rhythm | 13 | up | yes, R² 0.165 | better | **kept** |
| drum ratio | 1 | up | yes | better | **kept** |
| split drum/instrument timbre | 80 | **best in project** | no, ceiling flat | worse | dropped |
| the other 6 timbre families | 84 | up | — | much worse | dropped |
| energy (`rmse`, `zcr`) | 14 | up | no | worse | dropped |
| key-invariant chroma | 252 | flat | unclear | much worse | dropped |

Four of the six rejected candidates *raised* genre agreement, and one gave the
best genre number in the project's history. That column is the argument for
doing this the way round: a feature earns its place by whether a listener hears
it, and the frozen metrics are a check that nothing broke.

Key-invariant chroma is the one worth a sentence, because the hypothesis was
specific and it failed cleanly. Chroma measures absolute pitch, so a track and
its transposition look unrelated where a listener hears the same harmony;
rotating each vector to its strongest pitch class removes the key. It did not
help — rotated chroma correlates +0.030 with the human scores against +0.018 raw
— so absolute key is not why harmony fails. A likelier reason is that these are
histograms over thirty seconds, and C→F→G and G→F→C share a histogram while
sounding nothing alike.

## Still outstanding

- **A second round.** Forced choice, `heard` against `heard_wide`, new seeds,
  and the threshold for calling it a failure written down before it runs. Round
  one selected a candidate; it cannot also validate it.
- **`mood`**, the most used word in the vocabulary, with no feature behind it.
  Deliberately so: it is two raters' habit, and of 50 uses only 9 stood alone.
- **Same-artist exclusion**, which the other baseline applies and this one does
  not. It accounts for the entire gap between the two — 0.343 against 0.414 on
  the same data and the same features — and adding it would move every number in
  the README by about 0.07. That makes it a decision about the frozen metrics,
  not a bug fix.
- **The "why" the README promises.** No code exists for it. The vocabulary
  above is the natural language for it, but the tag analysis did not survive a
  control, so any explanation has to be phrased as geometry ("these two are in
  the closest 10% on harmonic features") rather than perception ("these two are
  connected by harmony").